# Task 3 — Training, Tuning & Regularisation

**Framework:** TensorFlow/Keras | **Seed:** 42

## Setup — data loading and model builders

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import random, os

np.random.seed(42); tf.random.set_seed(42); random.seed(42)
os.environ['PYTHONHASHSEED'] = '42'

from tensorflow.keras.datasets import mnist, cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models

# ── load + preprocess mnist ──
(mx_tr, my_tr), (mx_te, my_te) = mnist.load_data()
mx_tr = mx_tr.astype('float32')/255.0; mx_te = mx_te.astype('float32')/255.0
mx_tr = mx_tr.reshape(-1,28,28,1);     mx_te = mx_te.reshape(-1,28,28,1)
my_tr_oh = to_categorical(my_tr, 10);  my_te_oh = to_categorical(my_te, 10)

# ── load + preprocess cifar10 ──
(cx_tr, cy_tr), (cx_te, cy_te) = cifar10.load_data()
cx_tr = cx_tr.astype('float32')/255.0; cx_te = cx_te.astype('float32')/255.0
cy_tr_oh = to_categorical(cy_tr.flatten(), 10)
cy_te_oh  = to_categorical(cy_te.flatten(), 10)

cifar_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

# ── model builders ──
def build_lenet():
    m = models.Sequential([
        layers.Conv2D(6,  (5,5), padding='valid', activation='tanh', input_shape=(28,28,1)),
        layers.AveragePooling2D((2,2), strides=2),
        layers.Conv2D(16, (5,5), padding='valid', activation='tanh'),
        layers.AveragePooling2D((2,2), strides=2),
        layers.Flatten(),
        layers.Dense(120, activation='tanh'),
        layers.Dense(84,  activation='tanh'),
        layers.Dense(10,  activation='softmax'),
    ], name='lenet5')
    return m

def build_custom():
    m = models.Sequential([
        layers.Conv2D(32,  (3,3), padding='same', input_shape=(32,32,3)),
        layers.BatchNormalization(), layers.ReLU(), layers.MaxPooling2D(),
        layers.Conv2D(64,  (3,3), padding='same'),
        layers.BatchNormalization(), layers.ReLU(), layers.MaxPooling2D(),
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(), layers.ReLU(), layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax'),
    ], name='custom_cnn')
    return m

print("data and model builders ready")

## Problem 1 — first training run (LeNet-5, MNIST, SGD)

In [ ]:
# ── Problem 1: first training run — lenet on mnist ──

tf.random.set_seed(42); np.random.seed(42)
lenet = build_lenet()
lenet.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

hist = lenet.fit(
    mx_tr, my_tr_oh,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# loss curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist.history['loss'],     label='train loss')
ax1.plot(hist.history['val_loss'], label='val loss')
ax1.set_title('LeNet-5 Loss — SGD lr=0.01')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend()

ax2.plot(hist.history['accuracy'],     label='train acc')
ax2.plot(hist.history['val_accuracy'], label='val acc')
ax2.set_title('LeNet-5 Accuracy — SGD lr=0.01')
ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig('lenet_sgd_curves.png', dpi=100)
plt.show()

# overfitting detection
vl = hist.history['val_loss']
for ep in range(1, len(vl)):
    if vl[ep] > vl[ep-1]:
        print(f"overfitting starts around epoch {ep+1}")
        break

# test accuracy
loss, acc = lenet.evaluate(mx_te, my_te_oh, verbose=0)
print(f"final test accuracy: {acc:.4f}")

## Problem 2 — optimiser comparison

In [ ]:
# ── Problem 2: optimiser comparison ──

opts = {
    'SGD (no mom)': tf.keras.optimizers.SGD(learning_rate=0.01),
    'SGD+momentum': tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'Adam':         tf.keras.optimizers.Adam(learning_rate=0.001),
}

fig, ax = plt.subplots(figsize=(8, 5))
for name, opt in opts.items():
    tf.random.set_seed(42); np.random.seed(42)
    m = build_lenet()
    m.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(mx_tr, my_tr_oh, epochs=15, batch_size=64,
              validation_split=0.1, verbose=0)
    ax.plot(h.history['val_accuracy'], label=name)

ax.set_title('Optimiser Comparison — val accuracy (15 epochs, MNIST)')
ax.set_xlabel('epoch'); ax.set_ylabel('val accuracy'); ax.legend()
plt.tight_layout()
plt.savefig('optimiser_comparison.png', dpi=100)
plt.show()

# observation: adam converges fastest (adaptive lr per param).
# sgd+momentum reaches better final acc than plain sgd bcoz momentum
# smooths out gradient oscillations. plain sgd slowest + lowest acc.

## Problem 3 — LR × batch size grid search (CIFAR-10)

In [ ]:
# ── Problem 3: LR x batch size grid search (6 runs, cifar10 custom cnn) ──

lrs    = [0.1, 0.01, 0.001]
bsizes = [32, 128]
results = {}   # (lr, bs) -> val_acc

for lr in lrs:
    for bs in bsizes:
        tf.random.set_seed(42); np.random.seed(42)
        m = build_custom()
        # sgd+momentum chosen so lr effect is visible (adam masks it via adaptation)
        m.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9),
            loss='categorical_crossentropy', metrics=['accuracy']
        )
        h = m.fit(cx_tr, cy_tr_oh, epochs=10, batch_size=bs,
                  validation_split=0.1, verbose=0)
        va = h.history['val_accuracy'][-1]
        results[(lr, bs)] = round(va, 4)
        print(f"lr={lr}, bs={bs} -> val_acc={va:.4f}")

print("\n     lr \ bs |  bs=32  | bs=128")
for lr in lrs:
    print(f"  lr={lr:5.3f}   | {results[(lr,32)]:.4f}  | {results[(lr,128)]:.4f}")

## Problem 4 — regularisation experiment (4 variants)

In [ ]:
# ── Problem 4: regularisation experiment — 4 variants, 20 epochs ──

def train_variant(name, has_drop, has_bn, epochs=20):
    tf.random.set_seed(42); np.random.seed(42)
    inp = layers.Input(shape=(32,32,3))
    x = layers.Conv2D(32, (3,3), padding='same')(inp)
    if has_bn: x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x); x = layers.MaxPooling2D()(x)
    if has_drop: x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(64, (3,3), padding='same')(x)
    if has_bn: x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x); x = layers.MaxPooling2D()(x)
    if has_drop: x = layers.Dropout(0.3)(x)

    x = layers.GlobalAveragePooling2D()(x)
    if has_drop: x = layers.Dropout(0.5)(x)
    out = layers.Dense(10, activation='softmax')(x)

    m = models.Model(inp, out, name=name)
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(cx_tr, cy_tr_oh, epochs=epochs, batch_size=64,
              validation_split=0.1, verbose=0)
    return h

variants = {
    'no reg':       train_variant('no_reg',    False, False),
    'dropout only': train_variant('drop_only', True,  False),
    'batchnorm':    train_variant('bn_only',   False, True),
    'drop+bn':      train_variant('drop_bn',   True,  True),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
print(f"{'variant':18s} | train acc | val acc | gap")
for i, (name, h) in enumerate(variants.items()):
    tr = h.history['accuracy'][-1]
    va = h.history['val_accuracy'][-1]
    gap = tr - va
    axes.flatten()[i].plot(h.history['accuracy'],     label='train')
    axes.flatten()[i].plot(h.history['val_accuracy'], label='val')
    axes.flatten()[i].set_title(name)
    axes.flatten()[i].set_xlabel('epoch'); axes.flatten()[i].legend()
    print(f"{name:18s} | {tr:.4f}    | {va:.4f}  | {gap:.4f}")

plt.suptitle('Regularisation Variants — Accuracy Curves')
plt.tight_layout()
plt.show()

## Problem 5 — LR scheduling

In [ ]:
# ── Problem 5: LR scheduling — ReduceLROnPlateau vs Cosine Annealing ──

EPOCHS = 30

def train_sched(sched_name):
    tf.random.set_seed(42); np.random.seed(42)
    m = build_custom()
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss='categorical_crossentropy', metrics=['accuracy'])

    if sched_name == 'plateau':
        sched_cb = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, verbose=0
        )
    else:  # cosine
        sched_cb = tf.keras.callbacks.LearningRateScheduler(
            lambda ep: 0.001 * 0.5 * (1 + np.cos(np.pi * ep / EPOCHS)), verbose=0
        )

    lr_log = []
    log_cb = tf.keras.callbacks.LambdaCallback(
        on_epoch_end=lambda ep, logs: lr_log.append(float(m.optimizer.learning_rate))
    )

    h = m.fit(cx_tr, cy_tr_oh, epochs=EPOCHS, batch_size=64,
              validation_split=0.1, callbacks=[sched_cb, log_cb], verbose=0)
    return h, lr_log

h_plat, lr_plat = train_sched('plateau')
h_cos,  lr_cos  = train_sched('cosine')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(lr_plat)
axes[0,0].set_title('ReduceLROnPlateau — LR vs epoch')
axes[0,0].set_xlabel('epoch'); axes[0,0].set_ylabel('lr')

axes[0,1].plot(h_plat.history['val_accuracy'])
axes[0,1].set_title('ReduceLROnPlateau — val accuracy')
axes[0,1].set_xlabel('epoch'); axes[0,1].set_ylabel('accuracy')

axes[1,0].plot(lr_cos)
axes[1,0].set_title('Cosine Annealing — LR vs epoch')
axes[1,0].set_xlabel('epoch'); axes[1,0].set_ylabel('lr')

axes[1,1].plot(h_cos.history['val_accuracy'])
axes[1,1].set_title('Cosine Annealing — val accuracy')
axes[1,1].set_xlabel('epoch'); axes[1,1].set_ylabel('accuracy')

plt.suptitle('LR Schedule Comparison')
plt.tight_layout()
plt.savefig('lr_schedule_comparison.png', dpi=100)
plt.show()

## Analysis Questions — Task 3

**Q1.** a very high lr like 1.0 takes massive steps in the loss landscape. loss landscapes have narrow curved valleys — huge steps overshoot the valley bottom and bounce to the other wall. this causes the loss to oscillate or even blow up to infinity rather than converge. the gradient itself might look fine (points toward minimum) but the step size is so large the update jumps past the minimum entirely.

**Q2.** from my grid: lr=0.001 + bs=32 worked best, lr=0.1 + bs=128 was worst. hypothesis: high lr + large batch is a bad combo because large batches give sharp gradient estimates (low noise) that expose the instability of a big step size, causing divergence. small lr + small batch gives many small noisy updates that regularise implicitly and explore the loss landscape better.

**Q3.** dropout is disabled at inference because we want deterministic, full-network predictions — dropping neurons randomly would give different answers each forward pass. to compensate: during training when dropout(p) is applied, surviving activations are scaled up by 1/(1-p) so expected output magnitude is preserved. at test time, all neurons are used at their original scale — no correction needed because the training scaling already accounted for it.

**Q4.** ReduceLROnPlateau: triggered reactively when val_loss stagnates for `patience` epochs. LR curve is step-wise (flat then sudden drops). good when you dont know total epochs or training dynamics in advance.
Cosine Annealing: predetermined schedule, smooth curve from max LR down to near-zero following cosine. better when you know exactly how many epochs youll train and want smooth, predictable decay without monitoring overhead.